****PART A: Data preparation****

In [ ]:
import sys
sys.path.append('../')

from src.dataload import load_data
from src.preprocessing import preprocess_data
from src.feature_engineering import create_features
from src.aggregation import create_daily_metrics
from src.merge import merge_datasets

import pandas as pd

In [ ]:
sentiment, trades = load_data(
    "../data/raw/fear_greed_index.csv",
    "../data/raw/historical_data.csv"
)

In [ ]:
print("Sentiment shape: ",sentiment.shape)
print("Trades shape: ", trades.shape)

print("\nMissing Values: ", sentiment.isnull().sum())
print("\nMissing Values: ",trades.isnull().sum())

print("\nDuplicates: ",sentiment.duplicated().sum())
print("Duplicates: ",trades.duplicated().sum())

In [ ]:
sentiment,trades = preprocess_data(sentiment,trades)

In [ ]:
trades=create_features(trades)

In [ ]:
daily_metrics = create_daily_metrics(trades)
daily_metrics.head()

In [ ]:
final_df = merge_datasets(daily_metrics, sentiment)
final_df.head()

In [ ]:
final_df.to_csv(
    "C:/Users/Ritika Sundar/OneDrive/Desktop/SNU/SEMESTER 6/primetrade/data/preprocessed/final_dataset.csv",
    index=False
)

In [ ]:
def simplify_sentiment(x):
    if 'Fear' in x:
        return 'Fear'
    else:
        return 'Greed'
final_df['sentiment']=final_df['classification'].apply(simplify_sentiment)

****PART B: Data Analysis****

1. Performance vs Sentiment

In [ ]:
perf_summary=final_df.groupby('sentiment').agg({
    'daily_pnl':['mean','std'],
    'win_rate':'mean'
}).reset_index()
perf_summary.columns=['sentiment','avg_pnl','pnl_volatality','avg_win_rate']
perf_summary

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
#PNL plot
sns.barplot(data=final_df,x='sentiment',y='daily_pnl')
plt.title("PNL vs Sentiment")
plt.savefig("../outputs/plots/pnl_vs_sentiment.png")
plt.show()

In [ ]:
#Win Rate
sns.boxplot(data=final_df,x='sentiment',y='win_rate')
plt.title("Win rate distribution")
plt.savefig("../outputs/plots/winrate.png")
plt.show()

Observation:
- Greed days show lower average PNL compared to Fear
- Win rate is greater during greed periods but inconsistent (observed from the whisker spread)

2. Behaviour vs Sentiment

In [ ]:
behaviour_summary=final_df.groupby('sentiment').agg({
    'num_trades':'mean',
    'avg_trade_size':'mean',
    'long_ratio':'mean'    
}).reset_index()

behaviour_summary

In [ ]:
#trade frequency
sns.barplot(data=final_df,x='sentiment',y='num_trades')
plt.title("Trades per day")
plt.savefig("../outputs/plots/trades_per_day.png")
plt.show()

In [ ]:
#trade size
sns.boxplot(data=final_df, x='sentiment', y='avg_trade_size')
plt.title("Trade size distribution")
plt.savefig("../outputs/plots/trade_size_distribution.png")
plt.show()

In [ ]:
#long bias
sns.barplot(data=final_df,x='sentiment',y='long_ratio')
plt.title("Long Ratio")
plt.savefig("../outputs/plots/long_ratio.png")
plt.show()

Observation:
- Number of trades per day increases with fear
- The distribution of trades is the same. But sometimes, traders get overconfident with Greed and make higher trades (the outliers are more for Greed)
- Markets would work better in Fear as the long ratio is higher when observed in the plot

3. Segmentation

i) High vs Low activity traders

In [ ]:
median_trades=final_df['num_trades'].median()
final_df['activity_group']=final_df['num_trades'].apply(
    lambda x: 'High Activity' if x>median_trades else 'Low Activity'
)

ii) Large vs Small Trade size

In [ ]:
median_size=final_df['avg_trade_size'].median()
final_df['size_group']=final_df['avg_trade_size'].apply(
    lambda x: 'Large Trades' if x>median_size else 'Small Trades'
)

4. Segment Analysis

In [ ]:
activity_perf=final_df.groupby(
    ['activity_group', 'sentiment']
)['daily_pnl'].mean().reset_index()

sns.barplot(data=activity_perf, x='activity_group', y='daily_pnl', hue='sentiment')
plt.title("PNL by Activity Level")
plt.savefig("../outputs/plots/pnl_by_activity.png")
plt.show()

In [ ]:
size_perf=final_df.groupby(
    ['size_group', 'sentiment']
)['daily_pnl'].mean().reset_index()

sns.barplot(data=size_perf, x='size_group', y='daily_pnl', hue='sentiment')
plt.title("PNL by Trade Size")
plt.savefig("../outputs/plots/pnl_by_tradesize.png")
plt.show()

****PART C: Actionable Output****

**OUTPUT 1:**

- Average PNL is higher during fear periods
- Traders are found making more trades per day in this period as well

STRATEGY:
- Increase the trading frequency during this period and actively make trades

**OUTPUT 2:**

- Win rate remains almost the same across fear and greed
- PNL varies drastically

STRATEGY:
- During fear periods, do not optimize the win rate
- Focus on the higher risk-reward ratio